# Excel vs ЦФТ: Красноярский / Калининградский РФ (апр–май–июнь 2026)

Цепочка как для СПб / Алтая:

1. **Отбор** примеров из Excel по каждому РФ (логика `spb_rf_clients_apr_jun_tariff_selection`).
2. **Комиссии ЦФТ** по SQL коллеги (`DOG_OPER` + `VID_COMISS`).
3. **Сводка расхождений** Excel `Комиссия (₽ в месяц)` vs суммы `c_calc_summ` / `c_pay_summ` из озера.

Фильтр филиала: `contains` по `TARGET_RFS` (без учёта регистра).

Выход: `/home/jovyan/documents/Equaring/Data/rf_excel_vs_cft_krasn_kalin_apr_jun/`



In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


In [ ]:
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'rf_excel_vs_cft_krasn_kalin_apr_jun'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RFS = ['Красноярский', 'Калининградский']  # contains, case-insensitive
REG_DATE_MAX = pd.Timestamp('2026-01-01')
SAMPLE_PER_TARIFF = 5
STANDARD_TRX_SUM_MAX = 400_000.0
MENU_TRX_SUM_MAX = 0.0
COMM_MONTHLY_EPS = 0.005
COMPARE_ABS_TOL = 0.01
FILTER_MONTH = '2026-06'
MONTHS = ['2026-04', '2026-05', '2026-06']
TARIFF_ORDER = ['0', 'Меню возможностей', 'По Акту индивидуальный', 'Стандарт']
MONTH_COL_RU = {'2026-04': 'апрель', '2026-05': 'май', '2026-06': 'июнь'}
SHEET_NAME_BY_MONTH = {m: m for m in MONTHS}

excel_sources = [
    {'report_month': '2026-04-01', 'path': DATA_DIR / '04_Апрель_2026.xlsx', 'header': 0},
    {'report_month': '2026-05-01', 'path': DATA_DIR / '05_Май_2026.xlsx', 'header': 0},
    {'report_month': '2026-06-01', 'path': DATA_DIR / '06_Июнь_2026.xlsx', 'header': 0},
]

def rf_slug(name: str) -> str:
    mapping = {
        'Красноярский': 'krasnoyarsk',
        'Калининградский': 'kaliningrad',
    }
    return mapping.get(name, re.sub(r'[^a-zA-Z0-9а-яА-Я]+', '_', name).strip('_').lower())

for src in excel_sources:
    p = Path(src['path'])
    print(f"{src['report_month'][:7]}: exists={p.exists()} | {p}")
print('OUT_DIR:', OUT_DIR)
print('TARGET_RFS:', TARGET_RFS)


## 1) Helpers (как в SPB)


In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    cols = list(columns)
    norm_map = {normalize_colname(c): c for c in cols}
    for alias in aliases:
        if alias in cols:
            return alias
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    for alias in aliases:
        key = normalize_colname(alias)
        for nk, original in norm_map.items():
            if key and key in nk:
                return original
    return None


def to_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce',
    )


def normalize_agr_id(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    if re.fullmatch(r'\d+', s):
        return s
    return s or np.nan


def normalize_inn(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    s = re.sub(r'\D', '', s)
    return s or np.nan


def is_empty_date(series):
    dt = pd.to_datetime(series, errors='coerce')
    empty = series.isna() | (series.astype(str).str.strip().str.lower().isin(
        {'', 'nan', 'none', 'nat', 'na', '<null>', 'null', '-', '0', '0.0'}
    ))
    early = dt.notna() & (dt.dt.year <= 1901)
    return empty | early


def classify_tariff(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    sl = s.lower()
    if s == '0' or sl in {'0', '0.0', 'zero'}:
        return '0'
    if 'меню возможностей' in sl or sl.startswith('меню'):
        return 'Меню возможностей'
    if 'по акту' in sl or 'индивидуальн' in sl:
        return 'По Акту индивидуальный'
    if 'стандарт' in sl:
        return 'Стандарт'
    return None


COLUMN_ALIASES = {
    'agr_id': ['ID договора', 'agr_id', 'ИД договора', 'id договора'],
    'company_name': ['Наименование', 'Наименование клиента', 'company_name', 'Клиент'],
    'inn': ['ИНН', 'ИНН клиента', 'inn'],
    'contract_number': ['Номер договора', 'contract_number', '№ договора'],
    'd_valid_from': ['Дата регистрации договора', 'Дата заключения', 'd_valid_from', 'Дата регистрации'],
    'd_valid_to': ['Дата закрытия договора', 'Дата закрытия', 'd_valid_to', 'Дата расторжения'],
    'tariff': ['Тариф', 'tariff', 'Тарифный план'],
    'trx_sum': ['Сумма операций', 'trx_sum', 'Оборот'],
    'commission_from_ops': ['Комиссия (% с операций)', 'Комиссия % с операций', 'commission_from_ops', 'acq_pct'],
    'commission_monthly': ['Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'commission_monthly', 'Комиссия в месяц'],
    'filial': ['Филиал', 'Региональный филиал', 'Филиал договора', 'branch_nm', 'filial_rf'],
}


def resolve_columns(raw):
    resolved = {}
    for key, aliases in COLUMN_ALIASES.items():
        col = pick_column(raw.columns, aliases)
        if col is None and key not in ('d_valid_to', 'commission_from_ops', 'contract_number'):
            raise KeyError(f'Не найдена колонка {key}: aliases={aliases} | cols={list(raw.columns)[:40]}')
        resolved[key] = col
    return resolved


def load_excel_all():
    frames = []
    resolved_by_month = []
    for src in excel_sources:
        raw = pd.read_excel(src['path'], header=src['header'])
        resolved = resolve_columns(raw)
        df = pd.DataFrame({
            'agr_id': raw[resolved['agr_id']].map(normalize_agr_id),
            'company_name': raw[resolved['company_name']],
            'inn': raw[resolved['inn']].map(normalize_inn) if resolved['inn'] else np.nan,
            'contract_number': raw[resolved['contract_number']] if resolved['contract_number'] else np.nan,
            'd_valid_from': pd.to_datetime(raw[resolved['d_valid_from']], errors='coerce'),
            'd_valid_to': (
                pd.to_datetime(raw[resolved['d_valid_to']], errors='coerce')
                if resolved['d_valid_to'] else pd.NaT
            ),
            'tariff_raw': raw[resolved['tariff']],
            'trx_sum': to_num(raw[resolved['trx_sum']]),
            'commission_from_ops': (
                to_num(raw[resolved['commission_from_ops']])
                if resolved['commission_from_ops'] else np.nan
            ),
            'commission_monthly': to_num(raw[resolved['commission_monthly']]),
            'filial_raw': raw[resolved['filial']],
        })
        df['report_month'] = pd.to_datetime(src['report_month'])
        df['report_month_str'] = df['report_month'].dt.strftime('%Y-%m')
        df['tariff_group'] = df['tariff_raw'].map(classify_tariff)
        df['no_close_date'] = is_empty_date(df['d_valid_to'])
        df['filial_norm'] = (
            df['filial_raw'].astype(str)
            .str.replace('\xa0', ' ', regex=False)
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
        )
        df['filial_rf'] = df['filial_norm'].map(
            lambda v: re.match(r'^(.*?РФ)', v).group(1) if isinstance(v, str) and 'РФ' in v else v
        )
        frames.append(df)
        resolved_by_month.append({
            'month': src['report_month'][:7],
            **{k: v for k, v in resolved.items()},
            'rows_raw': len(raw),
        })
        print(f"{src['report_month'][:7]}: raw={len(raw):,}")
    excel_all = pd.concat(frames, ignore_index=True)
    display(pd.DataFrame(resolved_by_month))
    print('Total Excel rows:', len(excel_all))
    return excel_all


excel_all = load_excel_all()
print('Sample filial_rf:')
display(excel_all['filial_rf'].value_counts(dropna=False).head(30))


## 2) Отбор клиентов по каждому РФ

Правила как в SPB: все 3 месяца, стабильный тарифный сегмент, июньские фильтры для Стандарт / Меню, TOP-5 по комиссии для `0` и «По Акту».


In [ ]:
def _num0(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0


def _is_closed(v):
    if pd.isna(v):
        return False
    dt = pd.to_datetime(v, errors='coerce')
    if pd.notna(dt):
        if dt.year <= 1901:
            return False
        return True
    s = str(v).strip().lower()
    return s not in {'', 'nan', 'none', 'nat', 'na', '<null>', 'null', '-', '0', '0.0'}


def june_agr_frame(pool, month=FILTER_MONTH):
    g = pool[pool['report_month_str'] == month].copy()
    if g.empty:
        return pd.DataFrame(columns=[
            'agr_id', 'd_valid_from', 'has_close_date', 'trx_sum', 'commission_monthly',
            'tariff_raw', 'company_name', 'inn', 'contract_number', 'tariff_group',
        ])
    return (
        g.groupby('agr_id', as_index=False)
        .agg(
            d_valid_from=('d_valid_from', 'min'),
            has_close_date=('d_valid_to', lambda s: any(_is_closed(x) for x in s)),
            trx_sum=('trx_sum', lambda s: float(np.nansum([_num0(x) for x in s]))),
            commission_monthly=('commission_monthly', lambda s: float(np.nansum([_num0(x) for x in s]))),
            tariff_raw=('tariff_raw', 'first'),
            company_name=('company_name', 'first'),
            inn=('inn', 'first'),
            contract_number=('contract_number', 'first'),
            tariff_group=('tariff_group', 'first'),
        )
    )


def diagnose_june_filters(june_df, tariff_label, trx_max, trx_min_exclusive=None):
    n0 = len(june_df)
    m_reg = june_df['d_valid_from'].notna() & (june_df['d_valid_from'] <= REG_DATE_MAX)
    m_open = ~june_df['has_close_date'].astype(bool)
    trx = june_df['trx_sum'].map(_num0)
    m_trx = trx <= trx_max
    if trx_min_exclusive is not None:
        m_trx = m_trx & (trx > trx_min_exclusive)
    m_cm = june_df['commission_monthly'].map(_num0).abs() <= COMM_MONTHLY_EPS
    funnel = pd.DataFrame([
        {'step': 'июнь, уникальные agr_id', 'cnt': n0},
        {'step': f'd_valid_from <= {REG_DATE_MAX.date()}', 'cnt': int(m_reg.sum())},
        {'step': 'нет даты закрытия', 'cnt': int((m_reg & m_open).sum())},
        {'step': 'trx filter', 'cnt': int((m_reg & m_open & m_trx).sum())},
        {'step': 'комиссия ₽/мес ≈ 0', 'cnt': int((m_reg & m_open & m_trx & m_cm).sum())},
    ])
    print(f'\n=== Воронка {tariff_label} (июнь) ===')
    display(funnel)
    return set(june_df.loc[m_reg & m_open & m_trx & m_cm, 'agr_id'])


def build_agr_metrics(df):
    return (
        df.groupby('agr_id', as_index=False)
        .agg(
            tariff_group=('tariff_group', 'first'),
            tariff_raw=('tariff_raw', 'first'),
            company_name=('company_name', 'first'),
            inn=('inn', 'first'),
            commission_monthly_max=('commission_monthly', 'max'),
            commission_monthly_sum=('commission_monthly', 'sum'),
            trx_sum_max=('trx_sum', 'max'),
        )
    )


def select_for_rf(target_rf: str):
    mask_rf = excel_all['filial_norm'].str.contains(target_rf, case=False, na=False)
    excel_rf = excel_all[mask_rf & excel_all['agr_id'].notna()].copy()
    print(f'\n######## RF={target_rf} | rows={len(excel_rf):,} | agr={excel_rf["agr_id"].nunique():,} ########')
    display(excel_rf['tariff_group'].value_counts(dropna=False))

    months_per_agr = (
        excel_rf.groupby('agr_id')['report_month_str'].nunique().rename('months_cnt').reset_index()
    )
    stable_ids = set(months_per_agr.loc[months_per_agr['months_cnt'] == len(MONTHS), 'agr_id'])
    stable_df = excel_rf[excel_rf['agr_id'].isin(stable_ids)].copy()
    group_nunique = stable_df.groupby('agr_id')['tariff_group'].nunique(dropna=True)
    stable_tariff_ids = set(group_nunique[group_nunique == 1].index)
    has_group = (
        stable_df[stable_df['agr_id'].isin(stable_tariff_ids)]
        .groupby('agr_id')['tariff_group'].first()
    )
    has_group = has_group[has_group.isin(TARIFF_ORDER)]
    candidate_df = stable_df[stable_df['agr_id'].isin(set(has_group.index))].copy()
    candidate_df['tariff_group'] = candidate_df['agr_id'].map(has_group)
    print(f'Во всех {len(MONTHS)} месяцах: {len(stable_ids):,}')
    print(f'Стабильный tariff_group: {candidate_df["agr_id"].nunique():,}')

    selected_ids = []
    selection_log = []
    for tariff in TARIFF_ORDER:
        pool = candidate_df[candidate_df['tariff_group'] == tariff].copy()
        if pool.empty:
            selection_log.append({
                'target_rf': target_rf, 'tariff_group': tariff,
                'candidates': 0, 'selected': 0, 'note': 'пустой пул',
            })
            continue
        if tariff == 'Стандарт':
            june_df = june_agr_frame(pool)
            ok_ids = diagnose_june_filters(june_df, tariff, STANDARD_TRX_SUM_MAX, trx_min_exclusive=0.0)
            metrics = build_agr_metrics(pool[pool['agr_id'].isin(ok_ids)])
            june_trx = june_df.loc[june_df['agr_id'].isin(ok_ids), ['agr_id', 'trx_sum']].rename(
                columns={'trx_sum': 'trx_sum_june'}
            )
            metrics = metrics.merge(june_trx, on='agr_id', how='left')
            metrics = metrics.sort_values(['trx_sum_june', 'agr_id'], ascending=[True, True])
        elif tariff == 'Меню возможностей':
            june_df = june_agr_frame(pool)
            ok_ids = diagnose_june_filters(june_df, tariff, MENU_TRX_SUM_MAX)
            metrics = build_agr_metrics(pool[pool['agr_id'].isin(ok_ids)])
            metrics = metrics.sort_values(['agr_id'], ascending=[True])
        else:
            metrics = build_agr_metrics(pool)
            metrics = metrics.sort_values(
                ['commission_monthly_max', 'commission_monthly_sum', 'agr_id'],
                ascending=[False, False, True],
            )
        top = metrics.head(SAMPLE_PER_TARIFF).copy()
        selected_ids.extend(top['agr_id'].tolist())
        selection_log.append({
            'target_rf': target_rf,
            'tariff_group': tariff,
            'candidates': int(metrics['agr_id'].nunique()) if len(metrics) else 0,
            'selected': int(len(top)),
            'note': ', '.join(top['agr_id'].astype(str).tolist()) if len(top) else 'нет кандидатов',
        })
        print(f'=== {tariff}: candidates={selection_log[-1]["candidates"]} selected={selection_log[-1]["selected"]} ===')
        if len(top):
            display(top)

    if not selected_ids:
        print(f'WARN: RF={target_rf}: не удалось отобрать клиентов')
        return {
            'target_rf': target_rf,
            'selected_ids': [],
            'selection_log_df': pd.DataFrame(selection_log),
            'result_df': pd.DataFrame(),
            'summary_5col': pd.DataFrame(),
            'qc_df': pd.DataFrame(),
            'sheets_by_month': {},
        }

    detail = candidate_df[candidate_df['agr_id'].isin(selected_ids)].copy()
    result_df = detail[[
        'report_month_str', 'tariff_group', 'filial_rf',
        'agr_id', 'company_name', 'inn', 'contract_number',
        'd_valid_from', 'd_valid_to', 'tariff_raw',
        'trx_sum', 'commission_from_ops', 'commission_monthly',
    ]].copy()
    result_df['tariff_group'] = pd.Categorical(result_df['tariff_group'], categories=TARIFF_ORDER, ordered=True)
    agr_order = {agr: i for i, agr in enumerate(selected_ids)}
    result_df['_agr_order'] = result_df['agr_id'].map(agr_order)
    result_df = result_df.sort_values(['tariff_group', '_agr_order', 'report_month_str']).reset_index(drop=True)

    SHOW_COLS = [
        'Сегмент тарифа', 'ID договора', 'Наименование', 'ИНН', 'Номер договора',
        'Дата регистрации договора', 'Тариф', 'Сумма операций',
        'Комиссия (% с операций)', 'Комиссия (₽ в месяц)',
    ]
    result_cols_map = {
        'agr_id': 'ID договора', 'company_name': 'Наименование', 'inn': 'ИНН',
        'contract_number': 'Номер договора', 'd_valid_from': 'Дата регистрации договора',
        'tariff_raw': 'Тариф', 'trx_sum': 'Сумма операций',
        'commission_from_ops': 'Комиссия (% с операций)',
        'commission_monthly': 'Комиссия (₽ в месяц)',
    }
    sheets_by_month = {}
    for month in MONTHS:
        part = result_df[result_df['report_month_str'] == month].copy()
        show = part.rename(columns={'tariff_group': 'Сегмент тарифа', **result_cols_map})
        show = show.sort_values(['Сегмент тарифа', '_agr_order']).reset_index(drop=True)
        sheets_by_month[month] = show[SHOW_COLS].copy()

    pivot_src = result_df.copy()
    pivot_src['month_ru'] = pivot_src['report_month_str'].map(MONTH_COL_RU)
    pivot_num = pivot_src.groupby(['agr_id', 'month_ru'], as_index=False)['commission_monthly'].sum()
    wide = pivot_num.pivot(index='agr_id', columns='month_ru', values='commission_monthly')
    wide = wide.reindex(columns=['апрель', 'май', 'июнь'])
    tariff_map = pivot_src.sort_values(['agr_id', 'report_month_str']).groupby('agr_id')['tariff_raw'].first()
    segment_map = pivot_src.groupby('agr_id')['tariff_group'].first()
    summary_5col = (
        wide.reset_index()
        .assign(Тариф=lambda d: d['agr_id'].map(tariff_map))
        [['agr_id', 'Тариф', 'апрель', 'май', 'июнь']]
    )
    summary_5col['_ord'] = summary_5col['agr_id'].map(agr_order)
    summary_5col['_seg'] = summary_5col['agr_id'].map(segment_map)
    summary_5col['_seg'] = pd.Categorical(summary_5col['_seg'], categories=TARIFF_ORDER, ordered=True)
    summary_5col = summary_5col.sort_values(['_seg', '_ord']).drop(columns=['_ord', '_seg']).reset_index(drop=True)

    qc_base = result_df.copy()
    qc_base['tariff_group'] = qc_base['tariff_group'].astype(str)
    qc_df = (
        qc_base.groupby(['tariff_group', 'agr_id'], as_index=False, observed=True)
        .agg(
            months_cnt=('report_month_str', 'nunique'),
            commission_monthly_max=('commission_monthly', 'max'),
            commission_monthly_sum=('commission_monthly', 'sum'),
            trx_sum_max=('trx_sum', 'max'),
        )
    )

    slug = rf_slug(target_rf)
    out_xlsx = OUT_DIR / f'{slug}_selected_clients_apr_may_jun_2026.xlsx'
    out_summary_csv = OUT_DIR / f'{slug}_summary_5col.csv'
    selection_log_df = pd.DataFrame(selection_log)
    with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
        for month in MONTHS:
            sheets_by_month[month].to_excel(writer, sheet_name=SHEET_NAME_BY_MONTH[month], index=False)
        summary_5col.to_excel(writer, sheet_name='summary_5col', index=False)
        selection_log_df.to_excel(writer, sheet_name='selection_log', index=False)
        qc_df.to_excel(writer, sheet_name='qc_selected_agr', index=False)
    summary_5col.to_csv(out_summary_csv, index=False, encoding='utf-8-sig')
    print('Saved:', out_xlsx)
    print('Saved:', out_summary_csv)
    display(summary_5col)
    display(selection_log_df)

    return {
        'target_rf': target_rf,
        'selected_ids': selected_ids,
        'selection_log_df': selection_log_df,
        'result_df': result_df,
        'summary_5col': summary_5col,
        'qc_df': qc_df,
        'sheets_by_month': sheets_by_month,
    }


rf_results = {}
for rf in TARGET_RFS:
    rf_results[rf] = select_for_rf(rf)

selection_log_all = pd.concat(
    [r['selection_log_df'] for r in rf_results.values() if len(r['selection_log_df'])],
    ignore_index=True,
)
selected_meta_rows = []
for rf, r in rf_results.items():
    if not r['selected_ids']:
        continue
    meta = r['result_df'].sort_values('report_month_str').drop_duplicates('agr_id')[
        ['agr_id', 'tariff_group', 'tariff_raw', 'company_name', 'inn', 'filial_rf']
    ].copy()
    meta['target_rf'] = rf
    selected_meta_rows.append(meta)

selected_meta = (
    pd.concat(selected_meta_rows, ignore_index=True)
    if selected_meta_rows else pd.DataFrame(columns=['agr_id', 'target_rf'])
)
ALL_AGR_IDS = sorted({str(a) for r in rf_results.values() for a in r['selected_ids']})
print('ALL selected agr_id:', len(ALL_AGR_IDS))
print(ALL_AGR_IDS)
display(selection_log_all)
if not ALL_AGR_IDS:
    raise RuntimeError('Нет отобранных agr_id ни по одному РФ — дальше DOG_OPER запускать нельзя.')


## 3) Impala + SQL коллеги (DOG_OPER)

Источник: `ods.scd1_z_R2_IP_DOG_OPER` + `VID_COMISS` + `r2_ip_merchants` + `client`.
Фильтр: `o.c_date_create > '2026-03-31'`, `m.id in (отобранные agr_id)`.


In [ ]:
from rail_connectors.connection import connect

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp connection')

agr_ids_sql = ', '.join(str(int(x)) if str(x).isdigit() else f"'{x}'" for x in ALL_AGR_IDS)
print('agr_ids_sql count:', len(ALL_AGR_IDS))


In [ ]:
sql_commissions = f'''
select distinct
  m.c_cl_org cft_id,
  cl.c_name name_org,
  m.id id_agreement,
  m.c_name_in_pr agreement_num,
  m.c_date_begin,
  vc.id,
  vc.c_name commis_type,
  o.c_date_create,
  o.c_pay_summ,
  o.c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_R2_VID_COMISS vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and m.id in ({agr_ids_sql})
  and o.c_date_create > '2026-03-31'
order by o.c_date_create desc
'''

print(sql_commissions)

with imp:
    imp.execute('set MEM_LIMIT=8g')
    raw_df = imp.fetch(sql_commissions)

if raw_df is None:
    raw_df = pd.DataFrame()

print(f'raw rows: {len(raw_df):,}')
display(raw_df.head(50))


### Запасной SQL (lower-case имена таблиц)

Запускай **только если** предыдущая ячейка упала с «table not found».


In [ ]:
RUN_LOWERCASE_FALLBACK = False  # True, если основной SQL не нашёл таблицы

if RUN_LOWERCASE_FALLBACK:
    sql_commissions_lc = f'''
    select distinct
      m.c_cl_org cft_id,
      cl.c_name name_org,
      m.id id_agreement,
      m.c_name_in_pr agreement_num,
      m.c_date_begin,
      vc.id,
      vc.c_name commis_type,
      o.c_date_create,
      o.c_pay_summ,
      o.c_calc_summ
    from ods.scd1_z_r2_ip_dog_oper o
    join ods.scd1_z_r2_vid_comiss vc on vc.id = o.c_vid_comiss
    join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
    join ods.scd1_z_client cl on m.c_cl_org = cl.id
    where o.c_parent_class = 'R2_IP_MERCHANTS'
      and cl.class_id = 'CL_ORG'
      and m.id in ({agr_ids_sql})
      and o.c_date_create > '2026-03-31'
    order by o.c_date_create desc
    '''
    print(sql_commissions_lc)
    with imp:
        imp.execute('set MEM_LIMIT=8g')
        raw_df = imp.fetch(sql_commissions_lc)
    if raw_df is None:
        raw_df = pd.DataFrame()
    print(f'raw rows (lowercase): {len(raw_df):,}')
    display(raw_df.head(50))
else:
    print('SKIP lowercase fallback (RUN_LOWERCASE_FALLBACK=False)')


In [ ]:
# Агрегация DOG_OPER: id_agreement × месяц
if raw_df.empty:
    by_month_long = pd.DataFrame(columns=[
        'id_agreement', 'month', 'month_ru', 'rows', 'c_pay_summ', 'c_calc_summ'
    ])
    by_month_wide = pd.DataFrame()
    print('WARN: raw_df пустой — все отобранные agr_id без операций в ЦФТ после 2026-03-31')
else:
    agg_src = raw_df.copy()
    agg_src['id_agreement'] = agg_src['id_agreement'].astype(str).str.replace(r'\.0$', '', regex=True)
    agg_src['c_date_create'] = pd.to_datetime(agg_src['c_date_create'], errors='coerce')
    agg_src['month'] = agg_src['c_date_create'].dt.strftime('%Y-%m')
    agg_src['month_ru'] = agg_src['month'].map(MONTH_COL_RU)
    agg_src['c_pay_summ'] = pd.to_numeric(agg_src['c_pay_summ'], errors='coerce')
    agg_src['c_calc_summ'] = pd.to_numeric(agg_src['c_calc_summ'], errors='coerce')

    attrs = (
        agg_src.sort_values(['id_agreement', 'c_date_create'])
        .groupby('id_agreement', as_index=False)
        .agg(
            cft_id=('cft_id', 'first'),
            name_org=('name_org', 'first'),
            agreement_num=('agreement_num', 'first'),
            c_date_begin=('c_date_begin', 'first'),
        )
    )
    by_month_long = (
        agg_src.groupby(['id_agreement', 'month', 'month_ru'], as_index=False)
        .agg(
            rows=('c_date_create', 'size'),
            c_pay_summ=('c_pay_summ', 'sum'),
            c_calc_summ=('c_calc_summ', 'sum'),
        )
        .sort_values(['id_agreement', 'month'])
        .reset_index(drop=True)
    )
    pay_wide = (
        by_month_long.pivot(index='id_agreement', columns='month_ru', values='c_pay_summ')
        .reindex(columns=['апрель', 'май', 'июнь']).add_prefix('pay_')
    )
    calc_wide = (
        by_month_long.pivot(index='id_agreement', columns='month_ru', values='c_calc_summ')
        .reindex(columns=['апрель', 'май', 'июнь']).add_prefix('calc_')
    )
    by_month_wide = (
        attrs.set_index('id_agreement').join(pay_wide, how='left').join(calc_wide, how='left').reset_index()
    )
    order = {a: i for i, a in enumerate(ALL_AGR_IDS)}
    missing = [a for a in ALL_AGR_IDS if a not in set(by_month_wide['id_agreement'].astype(str))]
    if missing:
        by_month_wide = pd.concat([by_month_wide, pd.DataFrame([{'id_agreement': a} for a in missing])], ignore_index=True)
        print('Нет операций в DOG_OPER:', missing)
    by_month_wide['_ord'] = by_month_wide['id_agreement'].astype(str).map(order)
    by_month_wide = by_month_wide.sort_values('_ord').drop(columns=['_ord']).reset_index(drop=True)
    print('by_month_long / by_month_wide:')
    display(by_month_long.head(50))
    display(by_month_wide)

out_raw = OUT_DIR / 'lake_dog_oper_commissions_selected_raw.xlsx'
out_month = OUT_DIR / 'lake_dog_oper_commissions_selected_by_month.xlsx'
raw_df.to_excel(out_raw, index=False)
with pd.ExcelWriter(out_month, engine='openpyxl') as writer:
    by_month_wide.to_excel(writer, sheet_name='by_month_wide', index=False)
    by_month_long.to_excel(writer, sheet_name='by_month_long', index=False)
print('Saved:', out_raw)
print('Saved:', out_month)


## 4) Сравнение Excel vs ЦФТ

Ключ: `agr_id` × месяц.
Excel: `commission_monthly`. ЦФТ: `sum(c_calc_summ)`, `sum(c_pay_summ)`.
Расхождение: `|excel - c_calc| > 0.01` **или** нет строк в DOG_OPER за месяц.


In [ ]:
excel_long_parts = []
for rf, r in rf_results.items():
    if r['result_df'] is None or r['result_df'].empty:
        continue
    part = r['result_df'][[
        'agr_id', 'report_month_str', 'commission_monthly', 'trx_sum',
        'tariff_group', 'tariff_raw', 'company_name', 'inn', 'filial_rf',
    ]].copy()
    part['target_rf'] = rf
    part['agr_id'] = part['agr_id'].astype(str)
    excel_long_parts.append(part)

excel_long = pd.concat(excel_long_parts, ignore_index=True)
excel_month = (
    excel_long.groupby(['target_rf', 'agr_id', 'report_month_str'], as_index=False)
    .agg(
        excel_commission_monthly=('commission_monthly', 'sum'),
        excel_trx_sum=('trx_sum', 'sum'),
        tariff_group=('tariff_group', 'first'),
        tariff_raw=('tariff_raw', 'first'),
        company_name=('company_name', 'first'),
        inn=('inn', 'first'),
        filial_rf=('filial_rf', 'first'),
    )
)

if by_month_long.empty:
    cft_month = pd.DataFrame(columns=[
        'agr_id', 'report_month_str', 'cft_rows', 'cft_pay_summ', 'cft_calc_summ'
    ])
else:
    cft_month = by_month_long.rename(columns={
        'id_agreement': 'agr_id',
        'month': 'report_month_str',
        'rows': 'cft_rows',
        'c_pay_summ': 'cft_pay_summ',
        'c_calc_summ': 'cft_calc_summ',
    })[['agr_id', 'report_month_str', 'cft_rows', 'cft_pay_summ', 'cft_calc_summ']].copy()
    cft_month['agr_id'] = cft_month['agr_id'].astype(str)

compare_df = excel_month.merge(cft_month, on=['agr_id', 'report_month_str'], how='left')
compare_df['cft_rows'] = compare_df['cft_rows'].fillna(0).astype(int)
compare_df['cft_pay_summ'] = pd.to_numeric(compare_df['cft_pay_summ'], errors='coerce').fillna(0.0)
compare_df['cft_calc_summ'] = pd.to_numeric(compare_df['cft_calc_summ'], errors='coerce').fillna(0.0)
compare_df['excel_commission_monthly'] = pd.to_numeric(
    compare_df['excel_commission_monthly'], errors='coerce'
).fillna(0.0)
compare_df['delta_calc'] = compare_df['excel_commission_monthly'] - compare_df['cft_calc_summ']
compare_df['delta_pay'] = compare_df['excel_commission_monthly'] - compare_df['cft_pay_summ']
compare_df['ratio_calc'] = np.where(
    compare_df['cft_calc_summ'].abs() > COMPARE_ABS_TOL,
    compare_df['excel_commission_monthly'] / compare_df['cft_calc_summ'],
    np.nan,
)
compare_df['missing_in_cft'] = compare_df['cft_rows'] == 0
compare_df['is_discrepancy'] = (
    compare_df['missing_in_cft']
    | (compare_df['delta_calc'].abs() > COMPARE_ABS_TOL)
)

compare_df = compare_df.sort_values(
    ['target_rf', 'tariff_group', 'agr_id', 'report_month_str']
).reset_index(drop=True)
discrepancy_df = compare_df[compare_df['is_discrepancy']].copy()

print(f'compare rows: {len(compare_df):,} | discrepancies: {len(discrepancy_df):,}')
print('missing_in_cft months:', int(compare_df['missing_in_cft'].sum()))
display(compare_df.head(40))
display(discrepancy_df.head(40))

rf_summary = (
    compare_df.groupby('target_rf', as_index=False)
    .agg(
        agr_n=('agr_id', 'nunique'),
        month_rows=('agr_id', 'size'),
        disc_rows=('is_discrepancy', 'sum'),
        missing_cft_rows=('missing_in_cft', 'sum'),
    )
)
if len(selection_log_all):
    sel_sum = (
        selection_log_all.groupby('target_rf', as_index=False)
        .agg(selected_total=('selected', 'sum'), candidates_total=('candidates', 'sum'))
    )
    rf_summary = rf_summary.merge(sel_sum, on='target_rf', how='left')
display(rf_summary)

out_compare = OUT_DIR / 'compare_excel_vs_cft_by_month.xlsx'
out_disc = OUT_DIR / 'discrepancy_examples.xlsx'
out_sel_log = OUT_DIR / 'selection_log_by_rf.xlsx'

with pd.ExcelWriter(out_compare, engine='openpyxl') as writer:
    compare_df.to_excel(writer, sheet_name='compare', index=False)
    rf_summary.to_excel(writer, sheet_name='rf_summary', index=False)
    selection_log_all.to_excel(writer, sheet_name='selection_log', index=False)

with pd.ExcelWriter(out_disc, engine='openpyxl') as writer:
    discrepancy_df.to_excel(writer, sheet_name='discrepancies', index=False)
    compare_df[compare_df['missing_in_cft']].to_excel(writer, sheet_name='missing_in_cft', index=False)
    selected_meta.to_excel(writer, sheet_name='selected_meta', index=False)

with pd.ExcelWriter(out_sel_log, engine='openpyxl') as writer:
    selection_log_all.to_excel(writer, sheet_name='selection_log', index=False)
    rf_summary.to_excel(writer, sheet_name='rf_summary', index=False)

print('Saved:', out_compare)
print('Saved:', out_disc)
print('Saved:', out_sel_log)
print('OUT_DIR files:')
for p in sorted(OUT_DIR.glob('*')):
    print(' ', p.name)
